# 01 — Coleta e integração

**Dengue no Nordeste brasileiro — 2000–2025 · IMD1151, 2026.2**

Este notebook extrai ou reutiliza a base tratada do PostgreSQL e a integra à população e ao cadastro municipal do mesmo snapshot. A etapa termina com painéis agregados e verificações de integridade; não reestima testes nem altera as regras epidemiológicas.

Fontes originais: SINAN/OpenDataSUS e IBGE/SIDRA/Localidades.

O notebook anterior e a antiga etapa 02 foram arquivados fora do repositório. Consulte `docs/limpeza_repositorio.md` para sua localização. A pasta ativa contém o primeiro notebook do fluxo PostgreSQL.

**Reprodução portátil para a CP1:** quando não há uma extração individual disponível, o notebook usa automaticamente os agregados do SINAN e as tabelas do IBGE em `data/processed/insumos_portateis/`, incluídos no repositório. Esse modo dispensa banco, credenciais e dados individuais. Os totais e as regras são os mesmos do snapshot validado.

## 1. Ambiente e configuração

Execute com o ambiente `.venv` do repositório. As dependências do fluxo ativo estão reunidas em `requirements.txt`.

No modo `auto`, este notebook aproveita uma extração individual disponível e verifica seus hashes; sem esses arquivos, usa o pacote portátil versionado. `MODO_DADOS = "portatil"` força o pacote; `"postgresql"` exige a extração individual. Para consultar novamente o banco, use `REEXTRAIR = True` e configure `DENGUE_ENV` no ambiente, apontando para um `.env` privado, ou use `.env` na raiz. O banco precisa estar em execução. A conexão de extração é somente de leitura, com isolamento REPEATABLE READ. Senhas não são exibidas nem salvas no notebook.

In [1]:
from pathlib import Path
import importlib.metadata
import json
import os
import subprocess
import sys
import pandas as pd
from IPython.display import display

candidatos = [Path.cwd(), *Path.cwd().parents]
ROOT = next(p for p in candidatos if (p / 'scripts/extrair_postgresql.py').is_file())
sys.path.insert(0, str(ROOT / 'scripts'))
from integrar_postgresql import localizar_manifesto, conferir_arquivos, integrar, selecionar_entrada

REEXTRAIR = False
MODO_DADOS = os.environ.get("DENGUE_MODO_DADOS", "auto")
ARQUIVO_ENV = Path(os.environ.get('DENGUE_ENV', str(ROOT / '.env')))
print('Python:', sys.version.split()[0])
display(pd.DataFrame([{'biblioteca': nome, 'versao': importlib.metadata.version(nome)}
                      for nome in ['pandas', 'psycopg', 'python-dotenv', 'nbclient']]))

Python: 3.14.6


,biblioteca,versao
0,pandas,2.3.3
1,psycopg,3.3.9
2,python-dotenv,1.2.4
3,nbclient,0.11.0


## 2. Coleta de duas fontes e seleção da entrada

**Fonte 1 — SINAN/OpenDataSUS:** notificações da base tratada, por município de residência e ano. **Fonte 2 — IBGE:** população do SIDRA e cadastro de municípios/UFs da API de Localidades. As duas fontes são integradas nesta etapa, e suas URLs e transformações estão nos manifestos e nas especificações.

No modo PostgreSQL, os CSVs individuais são extraídos em conexão somente de leitura e ficam em `data/raw/postgresql/`, fora do Git. No modo portátil, são lidos os totais municipais do SINAN e as tabelas do IBGE versionados em `data/processed/insumos_portateis/`. São agregados de um snapshot real, sem registros individuais e sem nova consulta à rede ou ao banco.

O total de notificações, os casos prováveis e os confirmados são medidas distintas. A flag de caso provável continua sendo o numerador principal. Possíveis duplicatas são mantidas e contabilizadas. As fontes originais podem sofrer revisões; o pacote portátil congela a versão documentada.

In [2]:
if REEXTRAIR:
    if not ARQUIVO_ENV.is_file():
        raise FileNotFoundError('Configure DENGUE_ENV ou .env privado para extrair do PostgreSQL.')
    subprocess.run([sys.executable, '-B', str(ROOT / 'scripts/extrair_postgresql.py'),
                    '--env', str(ARQUIVO_ENV)], cwd=ROOT, check=True)
    MANIFESTO, modo = selecionar_entrada(ROOT, 'postgresql')
else:
    MANIFESTO, modo = selecionar_entrada(ROOT, MODO_DADOS)

print('Modo desta execução:', modo)
print('Manifesto:', MANIFESTO.relative_to(ROOT).as_posix())

Modo desta execução: pacote CP1 portatil: SINAN agregado + IBGE versionados, sem conexao ao banco
Manifesto: data/processed/insumos_portateis/manifesto.json


## 3. Proveniência, integridade e dicionário de dados

Os arquivos de entrada, o resumo anual e o dicionário são conferidos por SHA-256. No modo individual, as contagens do manifesto foram conciliadas com o PostgreSQL durante a extração. No modo portátil, o manifesto documenta a derivação dos agregados a partir dessa mesma extração validada.

Abaixo está o dicionário legível: nome, tipo, significado, origem e regra de cada variável. Ele também está disponível em `docs/dicionario_dados.md`, `.csv` e `.json`. Legendas epidemiológicas cuja confirmação oficial ainda está pendente permanecem identificadas como tal.

In [3]:
manifesto, arquivos = conferir_arquivos(ROOT, MANIFESTO)
display(pd.DataFrame([{
    'banco': manifesto['banco'], 'schema': manifesto['schema'],
    'periodo': '2000–2025', 'extraido_em': manifesto['concluido_em'],
    'somente_leitura_na_extracao': manifesto['somente_leitura'],
    'snapshot': manifesto['snapshot']
}]))
display(arquivos)
dicionario = pd.read_csv(ROOT / 'docs/dicionario_dados.csv')
tabela_sinan = ('notificacoes_municipio_ano' if manifesto.get('perfil') == 'portatil_agregado'
                else 'notificacao_dengue_ne')
with pd.option_context('display.max_colwidth', None):
    for tabela in [tabela_sinan, 'populacao_municipio_ano', 'populacao_ne',
                   'municipio_ne', 'estado_ne', 'paineis_integrados']:
        print('Dicionário:', tabela)
        display(dicionario.loc[dicionario['tabela'].eq(tabela),
                              ['variavel', 'tipo', 'significado', 'origem', 'regras_ou_unidade']])


,banco,schema,periodo,extraido_em,somente_leitura_na_extracao,snapshot
0,dengue-eda,public,2000–2025,2026-11-13T23:07:39.402694-03:00,True,902:902:


,tabela,registros,sha256_conferido
0,notificacoes_municipio_ano,40125,True
1,populacao_municipio_ano,46627,True
2,populacao_ne,43056,True
3,municipio_ne,1794,True
4,estado_ne,9,True


Dicionário:

 notificacoes_municipio_ano


,variavel,tipo,significado,origem,regras_ou_unidade
16,cod_mun7,texto,Município de residência,SINAN agregado da base tratada,Chave IBGE de sete dígitos; texto.
17,ano,inteiro,Ano de notificação,SINAN agregado da base tratada,2000–2025.
18,notificacoes,inteiro,Quantidade de notificações tratadas,Agregação da tabela tratada do SINAN,Inclui prováveis e não prováveis; unidade: registros.
19,casos_provaveis,inteiro,Quantidade com flag caso_provavel verdadeira,Agregação da tabela tratada do SINAN,Numerador principal; inclui possíveis duplicatas sinalizadas.
20,casos_confirmados,inteiro,Quantidade com flag caso_confirmado verdadeira,Agregação da tabela tratada do SINAN,Cenário de comparação; não substitui o numerador principal.
21,possiveis_duplicatas,inteiro,Quantidade com flag possivel_duplicata verdadeira,Agregação da tabela tratada do SINAN,"Contagem entre todas as notificações, não apenas casos prováveis."
22,sem_idade,inteiro,Notificações sem idade tratada,Agregação da tabela tratada do SINAN,Contagem entre todas as notificações; idade 0 é válida.
23,sem_sexo,inteiro,Notificações sem sexo tratado,Agregação da tabela tratada do SINAN,Nulo ou vazio; contagem entre todas as notificações.
24,sem_mes_sintomas,inteiro,Notificações sem mês de sintomas tratado,Agregação da tabela tratada do SINAN,Contagem entre todas as notificações.


Dicionário: populacao_municipio_ano


... (5 outputs left out)

Dicionário: estado_ne


,variavel,tipo,significado,origem,regras_ou_unidade
44,uf_cod,inteiro,Código IBGE da UF,IBGE/Localidades,Chave primária.
45,sigla,texto,Sigla da UF,IBGE/Localidades,"MA, PI, CE, RN, PB, PE, AL, SE, BA."
46,nome,texto,Nome da UF,IBGE/Localidades,Descrição da unidade federativa.
47,regiao_sigla,texto,Sigla da grande região,IBGE/Localidades,NE no recorte.
48,regiao,texto,Nome da grande região,IBGE/Localidades,Nordeste no recorte.


Dicionário: paineis_integrados


,variavel,tipo,significado,origem,regras_ou_unidade
49,notificacoes,inteiro,Quantidade de notificações tratadas,Soma dos agregados municipais,Inclui prováveis e não prováveis; unidade: registros.
50,casos_provaveis,inteiro,Quantidade com flag caso_provavel verdadeira,Soma dos agregados municipais,Numerador principal; inclui possíveis duplicatas sinalizadas.
51,casos_confirmados,inteiro,Quantidade com flag caso_confirmado verdadeira,Soma dos agregados municipais,Cenário de comparação; não substitui o numerador principal.
52,possiveis_duplicatas,inteiro,Quantidade com flag possivel_duplicata verdadeira,Soma dos agregados municipais,"Contagem entre todas as notificações, não apenas casos prováveis."
53,sem_idade,inteiro,Notificações sem idade tratada,Soma dos agregados municipais,Contagem entre todas as notificações; idade 0 é válida.
54,sem_sexo,inteiro,Notificações sem sexo tratado,Soma dos agregados municipais,Nulo ou vazio; contagem entre todas as notificações.
55,sem_mes_sintomas,inteiro,Notificações sem mês de sintomas tratado,Soma dos agregados municipais,Contagem entre todas as notificações.
56,populacao,"inteiro, habitantes","Denominador do município, UF ou região no ano",Integração SINAN + IBGE,Somar denominadores antes de calcular a taxa; não representa pessoas distintas ao somar anos.
57,incidencia_provaveis_100mil,número real,Incidência operacional anual por 100 mil habitantes,Integração SINAN + IBGE,"casos_provaveis / populacao × 100.000, com população positiva; não é média das taxas municipais."
58,fonte_populacao,texto,Origem do denominador municipal,Integração SINAN + IBGE,Preserva a informação de interpolação do banco.


## 4. Integração por município e ano

1. No modo individual, as notificações são lidas em blocos de 200 mil linhas e agregadas por `cod_mun7` e `ano`. No modo portátil, são lidos esses mesmos totais já agregados, cujas somas anuais são novamente conferidas.
2. Municípios e UFs são ligados por `uf_cod`, com validação de cardinalidade; notificações e população usam o código IBGE de sete dígitos e o mesmo ano.
3. O painel preserva municípios-ano com população e nenhum caso, preenchendo apenas as contagens com zero. A junção externa também conserva eventuais notificações sem denominador, que são auditadas.
4. A população já tratada do banco é mantida, incluindo a fonte `interpolada` em 2007 e 2023. Nenhum município recebe 26 anos de cobertura por pressuposto.
5. A incidência é `casos_provaveis / populacao × 100.000`, com denominador positivo. Possíveis duplicatas continuam incluídas e sinalizadas; esta etapa não as remove.

Os sete totais anuais são comparados ao resumo obtido diretamente do PostgreSQL. Não são exibidos registros individuais.

In [4]:
resultado = integrar(ROOT, MANIFESTO)
validacao = resultado['validacao']
print('Integração:', validacao['status'])
print('Comprovante:', resultado['caminho_validacao'])
display(pd.DataFrame([{'metrica': k, 'total': v} for k, v in validacao['totais'].items()]))

Modo portatil: 40,125 linhas municipais, representando 5,474,491 notificacoes.


Integração: concluido
Comprovante: data/metadata/postgresql/relatorio_2000_2025/portatil_20261113/integracoes/20261113_233631.json


,metrica,total
0,notificacoes,5474491
1,casos_provaveis,4178450
2,casos_confirmados,2884120
3,possiveis_duplicatas,101804
4,sem_idade,10816
5,sem_sexo,6682
6,sem_mes_sintomas,12928


## 5. Painéis integrados e cobertura

As tabelas abaixo são agregadas. O painel regional soma os numeradores e denominadores estaduais antes de calcular a taxa; não faz média simples das taxas municipais. A cobertura populacional pode variar conforme o ano e a criação dos municípios.

In [5]:
regiao = resultado['painel_regiao_ano']
ufs = resultado['painel_uf_ano']
cobertura = resultado['cobertura']
display(regiao[['ano', 'notificacoes', 'casos_provaveis', 'casos_confirmados',
                'populacao', 'incidencia_provaveis_100mil']])
display(cobertura.groupby('uf', as_index=False).agg(
    primeiro_ano=('ano', 'min'), ultimo_ano=('ano', 'max'),
    menor_cobertura=('municipios_com_populacao', 'min'),
    maior_cobertura=('municipios_com_populacao', 'max'),
    municipio_anos_interpolados=('municipios_interpolados', 'sum')))
display(ufs.loc[ufs['ano'].eq(2025), ['uf','ano','notificacoes','casos_provaveis',
                                     'populacao','incidencia_provaveis_100mil']])

,ano,notificacoes,casos_provaveis,casos_confirmados,populacao,incidencia_provaveis_100mil
0,2000,83283,70187,50929,47782486,146.888548
1,2001,180931,149734,95902,48331186,309.808247
2,2002,326105,267759,182581,48845112,548.179724
3,2003,207668,151541,109463,49352225,307.060117
4,2004,41850,22368,17025,50427274,44.356949
5,2005,124662,76039,61463,51019091,149.040288
6,2006,107714,69757,59183,51609027,135.164339
7,2007,175517,125608,91581,52345293,239.960449
8,2008,299716,209255,148698,53088496,394.162607
9,2009,177789,126763,83696,53591197,236.536982


,uf,primeiro_ano,ultimo_ano,menor_cobertura,maior_cobertura,municipio_anos_interpolados
0,AL,2000,2025,101,102,204
1,BA,2000,2025,415,417,834
2,CE,2000,2025,184,184,368
3,MA,2000,2025,217,217,434
4,PB,2000,2025,223,223,446
5,PE,2000,2025,185,185,370
6,PI,2000,2025,221,224,447
7,RN,2000,2025,166,167,334
8,SE,2000,2025,75,75,150


,uf,ano,notificacoes,casos_provaveis,populacao,incidencia_provaveis_100mil
25,MA,2025,5672,5672,7018211,80.818317
51,PI,2025,9289,9289,3384547,274.453272
77,CE,2025,5454,5454,9268836,58.842340
103,RN,2025,9480,9480,3455236,274.366208
129,PB,2025,7769,7769,4164468,186.554441
155,PE,2025,21050,21050,9562007,220.142069
181,AL,2025,8096,8096,3220848,251.362374
207,SE,2025,712,712,2299425,30.964263
233,BA,2025,32434,32434,14870907,218.103711


## 6. Reconciliação e arquivos de saída

A execução deve falhar se houver divergências de hashes, chaves duplicadas, registros fora do recorte, perda de contagens, duplicação de população ou casos prováveis sem denominador. As taxas mantêm a precisão numérica dos painéis; as tabelas exibidas podem arredondar apenas a apresentação.

In [6]:
assert validacao['totais'] == manifesto['totais']
assert validacao['casos_provaveis_sem_populacao'] == 0
assert len(regiao) == 26 and len(ufs) == 234
print('Notificações sem população:', validacao['notificacoes_sem_populacao'])
print('Casos prováveis sem população:', validacao['casos_provaveis_sem_populacao'])
display(pd.DataFrame([{'verificacao': v, 'resultado': 'aprovada'}
                      for v in validacao['verificacoes']]))
display(pd.DataFrame([{'tabela': k, **v} for k,v in validacao['arquivos'].items()]))

Notificações sem população: 0
Casos prováveis sem população: 0


,verificacao,resultado
0,SHA-256 das entradas,aprovada
1,chaves unicas,aprovada
2,juncoes validadas,aprovada
3,recorte temporal e geografico,aprovada
4,sete metricas por ano conciliadas com PostgreSQL,aprovada
5,populacao conservada na juncao,aprovada
6,26 anos e 234 UF-anos,aprovada
7,casos provaveis sem denominador igual a zero,aprovada


,tabela,arquivo,linhas,sha256
0,painel_municipio_ano,data/processed/postgresql/relatorio_2000_2025/...,46627,23101d31e040bff0df1a68eec6b542f74e648153e5ec53...
1,painel_uf_ano,data/processed/postgresql/relatorio_2000_2025/...,234,f5a99ef3fd55cbff9a8f0bda440a02621e8bfd38bb4ee6...
2,painel_regiao_ano,data/processed/postgresql/relatorio_2000_2025/...,26,b2d8d741c6ba54c587747016d571f7d3e9900e507820d9...
3,cobertura_populacao_uf_ano,data/processed/postgresql/relatorio_2000_2025/...,234,c2ea5731e12bc95b90afcec4b9bf24ca83d336ed51ee54...


## 7. Resultado desta etapa e limites

A coleta pelo PostgreSQL e a integração resultam em quatro tabelas: município-ano, UF-ano, região-ano e cobertura populacional. Cada execução grava uma pasta própria e um comprovante com hashes, preservando as anteriores. Os arquivos agregados e os metadados podem ser versionados; os registros individuais permanecem locais.

Esta etapa conserva o tratamento já aplicado no banco. Ela não valida externamente a definição de caso nem elimina subnotificação, mudanças da ficha do SINAN ou diferenças na investigação entre UFs. A incidência usa o numerador operacional do projeto e 2025 permanece sujeito a revisão. Licenças e decisões pendentes do grupo continuam abertas.

A EDA e os testes das perguntas P1–P4 pertencem às etapas posteriores. Consulte `database/reproducao/.claude/specs/03_decisoes.md`, `docs/extracao_postgresql.md` e o registro de IA em `.claude/specs/99_uso_ia.md`.

O modo portátil atende à reprodução da coleta/integração CP1 e dos painéis anuais. Ele não contém os registros necessários para novas distribuições individuais de idade, sexo ou mês; essas análises requerem a base individual. O modo PostgreSQL foi mantido para esse uso. Nenhuma decisão de tratamento ou de definição de caso foi modificada.